# Recommender Build

This notebook builds the recommendation pipeline for the **Personalized Content Recommendation Engine**.

It will:
- load the raw CSV files
- create processed outputs for modeling
- build a popularity baseline
- build a TF-IDF content-based recommender
- create a simple hybrid recommender
- save processed datasets and model artifacts for `app.py`


## 1. Imports and paths

In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import pickle
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display


In [2]:
cwd = Path.cwd()

if (cwd / 'data' / 'raw').exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / 'data' / 'raw').exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError('Could not find data/raw from the current notebook location.')

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR = PROJECT_ROOT / 'models'
FIG_DIR = PROJECT_ROOT / 'results' / 'figures'
METRICS_DIR = PROJECT_ROOT / 'results' / 'metrics'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Raw data path:', RAW_DIR)
print('Processed path:', PROCESSED_DIR)
print('Models path:', MODELS_DIR)


Project root: C:\Users\Owen\Documents\Hertsfordshire\Projects\personalized-content
Raw data path: C:\Users\Owen\Documents\Hertsfordshire\Projects\personalized-content\data\raw
Processed path: C:\Users\Owen\Documents\Hertsfordshire\Projects\personalized-content\data\processed
Models path: C:\Users\Owen\Documents\Hertsfordshire\Projects\personalized-content\models


## 2. Load raw data

In [3]:
users = pd.read_csv(RAW_DIR / 'users.csv')
content = pd.read_csv(RAW_DIR / 'content.csv')
interactions = pd.read_csv(RAW_DIR / 'interactions.csv')

interactions['interaction_timestamp'] = pd.to_datetime(interactions['interaction_timestamp'])

print('users shape:', users.shape)
print('content shape:', content.shape)
print('interactions shape:', interactions.shape)


users shape: (450, 19)
content shape: (900, 18)
interactions shape: (22000, 15)


## 3. Create processed datasets

In [4]:
# Content features dataset
content_features = content.copy()
text_cols = ['title', 'category', 'subcategory', 'tags', 'format', 'mood', 'depth_level']
content_features[text_cols] = content_features[text_cols].fillna('')
content_features['text_features'] = (
    content_features['title'].astype(str) + ' ' +
    content_features['category'].astype(str) + ' ' +
    content_features['subcategory'].astype(str) + ' ' +
    content_features['tags'].astype(str).str.replace('|', ' ', regex=False) + ' ' +
    content_features['format'].astype(str) + ' ' +
    content_features['mood'].astype(str) + ' ' +
    content_features['depth_level'].astype(str)
)

# User profiles dataset
user_profiles = users.copy()
user_profiles['favorite_category_list'] = user_profiles['favorite_categories'].fillna('').str.split('|')
user_profiles['preferred_format_list'] = user_profiles['preferred_formats'].fillna('').str.split('|')
user_profiles['preferred_mood_list'] = user_profiles['preferred_moods'].fillna('').str.split('|')

# Interaction scores dataset
interaction_scores = interactions.copy()
interaction_scores['rating_filled'] = interaction_scores['rating'].fillna(0)
interaction_scores['engagement_bucket'] = pd.cut(
    interaction_scores['engagement_score'],
    bins=[-0.01, 3, 6, 9, 20],
    labels=['Low', 'Medium', 'High', 'Very High']
)

display(content_features.head(2))
display(user_profiles.head(2))
display(interaction_scores.head(2))


,content_id,title,category,subcategory,tags,format,mood,depth_level,duration_minutes,cost_band,time_of_day_fit,location_scope,is_local_content,freshness_band,publish_date,popularity_score,recency_score,content_quality_score,text_features
0,C0001,Stretching Routines for Long Study Days for Be...,Fitness,Beginner Fitness,beginner|workout|gym|stretching,Video,Focused,Light,35,Low,Afternoon,Online,0,New,2025-09-28,71,19,73,Stretching Routines for Long Study Days for Be...
1,C0002,A Simple Gym Plan for Absolute Beginners for B...,Fitness,Beginner Fitness,fitness|beginner|stretching|workout,Guide,Focused,Moderate,12,Low,Weekend,Online,0,Evergreen,2025-10-22,65,31,92,A Simple Gym Plan for Absolute Beginners for B...


,user_id,age_group,study_level,study_mode,persona,favorite_categories,preferred_formats,preferred_moods,session_pattern,activity_level,...,campus_affinity,preferred_days,device_preference,home_region,discovery_style,avg_session_minutes,onboarding_interest_score,favorite_category_list,preferred_format_list,preferred_mood_list
0,U0001,21-23,Postgraduate Taught,Full-time,Pop Culture Browser,Events|Music,Video|Playlist,Relaxed|Curious,Morning,High,...,Mixed,Mixed,Laptop,Watford,Trendy,36,0.70,"[Events, Music]","[Video, Playlist]","[Relaxed, Curious]"
1,U0002,24-26,Undergraduate,Full-time,Career Builder,Tech|Finance|Career,Long Video|Article,Focused|Curious,Afternoon,Medium,...,Mixed,Mixed,Mobile,Remote/Online,Balanced,38,0.93,"[Tech, Finance, Career]","[Long Video, Article]","[Focused, Curious]"


,interaction_id,user_id,content_id,viewed,liked,saved,shared,completed,dwell_time_percent,rating,engaged_minutes,interaction_timestamp,engagement_score,interaction_day_type,recommendation_surface,rating_filled,engagement_bucket
0,I000001,U0133,C0103,1,0,0,0,0,72,NaN,23.0,2026-02-06 22:41:00,2.44,Weekday,Home Feed,0.0,Low
1,I000002,U0085,C0300,1,0,0,1,1,90,NaN,3.6,2025-09-06 08:06:00,8.30,Weekend,Trending,0.0,High


In [5]:
content_features.to_csv(PROCESSED_DIR / 'content_features.csv', index=False)
user_profiles.drop(columns=['favorite_category_list', 'preferred_format_list', 'preferred_mood_list']).to_csv(
    PROCESSED_DIR / 'user_profiles.csv', index=False
)
interaction_scores.to_csv(PROCESSED_DIR / 'interaction_scores.csv', index=False)

print('Saved processed CSV files.')


Saved processed CSV files.


## 4. Popularity baseline

In [6]:
content_popularity = (
    interaction_scores.groupby('content_id')
    .agg(
        total_views=('viewed', 'sum'),
        total_likes=('liked', 'sum'),
        total_saves=('saved', 'sum'),
        total_shares=('shared', 'sum'),
        total_completions=('completed', 'sum'),
        avg_engagement_score=('engagement_score', 'mean'),
        interaction_count=('interaction_id', 'count')
    )
    .reset_index()
)

content_popularity = content_popularity.merge(
    content[['content_id', 'title', 'category', 'format', 'popularity_score', 'recency_score', 'content_quality_score']],
    on='content_id',
    how='left'
)

content_popularity['popularity_rank_score'] = (
    0.30 * content_popularity['avg_engagement_score'] +
    0.20 * content_popularity['total_likes'] +
    0.15 * content_popularity['total_saves'] +
    0.10 * content_popularity['total_shares'] +
    0.10 * content_popularity['total_completions'] +
    0.10 * content_popularity['popularity_score'] +
    0.05 * content_popularity['recency_score']
)

top_popular = content_popularity.sort_values('popularity_rank_score', ascending=False).reset_index(drop=True)
top_popular.head(10)


,content_id,total_views,total_likes,total_saves,total_shares,total_completions,avg_engagement_score,interaction_count,title,category,format,popularity_score,recency_score,content_quality_score,popularity_rank_score
0,C0726,45,34,24,6,23,8.738333,48,A Smarter Student Money Routine for the Month ...,Finance,Guide,84,47,84,26.671500
1,C0292,30,19,16,10,19,9.428387,31,Interesting Workshops and Meetups to Check Out...,Events,Carousel,96,100,73,26.528516
2,C0525,40,25,18,7,24,8.602439,41,7 Revision Habits That Actually Stick for Grou...,Study,Article,85,75,77,25.630732
3,C0673,44,21,17,9,18,6.955833,48,Local Events That Feel Social Without Being To...,Events,Carousel,94,93,79,25.586750
4,C0216,45,24,21,7,22,8.096170,47,Budget-Friendly Food Spots Worth Trying This W...,Food,Article,92,61,91,25.528851
5,C0169,43,26,14,14,20,7.542500,48,Budget-Friendly Food Spots Worth Trying This W...,Food,Article,95,56,71,25.262750
6,C0244,45,29,11,11,17,6.766939,49,Budget-Friendly Food Spots Worth Trying This W...,Food,Short Video,88,74,95,24.780082
7,C0264,39,19,12,11,18,6.789091,44,Room Setup Changes That Make Studying Easier T...,Lifestyle,Short Video,99,85,65,24.686727
8,C0188,39,22,17,10,22,7.993333,42,Questions to Ask at Networking Events That Lea...,Career,Guide,86,64,79,24.348000
9,C0636,42,22,15,11,23,7.125417,48,Lifestyle Upgrades That Feel Small but Useful ...,Lifestyle,Guide,82,77,77,24.237625


## 5. Build TF-IDF content-based recommender

In [7]:
tfidf = TfidfVectorizer(stop_words='english', max_features=5000, ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(content_features['text_features'])
similarity_matrix = cosine_similarity(tfidf_matrix)

content_index = pd.Series(content_features.index, index=content_features['content_id']).drop_duplicates()

print('TF-IDF matrix shape:', tfidf_matrix.shape)
print('Similarity matrix shape:', similarity_matrix.shape)


TF-IDF matrix shape: (900, 2248)
Similarity matrix shape: (900, 900)


In [8]:
def get_similar_items(content_id, top_n=10):
    if content_id not in content_index.index:
        return pd.DataFrame()
    idx = content_index[content_id]
    sim_scores = list(enumerate(similarity_matrix[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1: top_n + 1]
    item_indices = [i[0] for i in sim_scores]
    item_scores = [i[1] for i in sim_scores]
    results = content_features.iloc[item_indices][['content_id', 'title', 'category', 'format', 'mood']].copy()
    results['similarity_score'] = item_scores
    return results

sample_content_id = content_features['content_id'].iloc[0]
get_similar_items(sample_content_id, top_n=5)


,content_id,title,category,format,mood,similarity_score
378,C0379,Stretching Routines for Long Study Days for Be...,Fitness,Article,Energetic,0.674539
448,C0449,Stretching Routines for Long Study Days Withou...,Fitness,Video,Energetic,0.544311
69,C0070,Stretching Routines for Long Study Days for Sm...,Fitness,Guide,Focused,0.530484
151,C0152,Stretching Routines for Long Study Days for Be...,Fitness,Article,Energetic,0.510842
250,C0251,Stretching Routines for Long Study Days for Sm...,Fitness,Short Video,Focused,0.467333


## 6. User preference profile from historical behavior

In [9]:
user_history = interaction_scores.merge(
    content[['content_id', 'title', 'category', 'format', 'mood']],
    on='content_id', how='left'
)

user_positive_history = user_history[
    (user_history['engagement_score'] >= 6) |
    (user_history['liked'] == 1) |
    (user_history['saved'] == 1) |
    (user_history['completed'] == 1)
].copy()

user_top_categories = (
    user_positive_history.groupby(['user_id', 'category'])['engagement_score']
    .mean()
    .reset_index()
    .sort_values(['user_id', 'engagement_score'], ascending=[True, False])
)

user_top_formats = (
    user_positive_history.groupby(['user_id', 'format'])['engagement_score']
    .mean()
    .reset_index()
    .sort_values(['user_id', 'engagement_score'], ascending=[True, False])
)

user_top_moods = (
    user_positive_history.groupby(['user_id', 'mood'])['engagement_score']
    .mean()
    .reset_index()
    .sort_values(['user_id', 'engagement_score'], ascending=[True, False])
)

display(user_top_categories.head(10))


,user_id,category,engagement_score
3,U0001,Music,10.806667
1,U0001,Events,8.865000
4,U0001,Tech,8.100000
6,U0001,Wellness,7.600000
5,U0001,Travel,7.346667
2,U0001,Finance,7.160000
0,U0001,Entertainment,6.993333
7,U0002,Career,10.836364
13,U0002,Tech,10.185455
10,U0002,Finance,9.740000


## 7. Hybrid recommender

In [10]:
content_popularity_lookup = top_popular[['content_id', 'popularity_rank_score']].copy()
max_pop_score = content_popularity_lookup['popularity_rank_score'].max()
content_popularity_lookup['normalized_popularity_score'] = content_popularity_lookup['popularity_rank_score'] / max_pop_score
content_popularity_lookup = content_popularity_lookup[['content_id', 'normalized_popularity_score']]

seen_lookup = interaction_scores.groupby('user_id')['content_id'].apply(set).to_dict()

def get_user_seed_items(user_id, min_score=6, top_k=3):
    user_rows = user_positive_history[user_positive_history['user_id'] == user_id].copy()
    if user_rows.empty:
        return []
    seeds = (
        user_rows.sort_values('engagement_score', ascending=False)
        [['content_id', 'engagement_score']]
        .drop_duplicates('content_id')
        .head(top_k)
    )
    return seeds['content_id'].tolist()

def recommend_for_user(user_id, top_n=10):
    seen_items = seen_lookup.get(user_id, set())
    seed_items = get_user_seed_items(user_id, top_k=3)

    if not seed_items:
        cold_start = top_popular.loc[~top_popular['content_id'].isin(seen_items)].head(top_n).copy()
        cold_start['recommendation_reason'] = 'Popularity fallback for new or low-history user'
        return cold_start[['content_id', 'title', 'category', 'format', 'popularity_rank_score', 'recommendation_reason']]

    candidate_frames = []
    for seed in seed_items:
        sims = get_similar_items(seed, top_n=50)
        if sims.empty:
            continue
        sims['seed_content_id'] = seed
        candidate_frames.append(sims)

    if not candidate_frames:
        fallback = top_popular.loc[~top_popular['content_id'].isin(seen_items)].head(top_n).copy()
        fallback['recommendation_reason'] = 'Popularity fallback because no similar content candidates were found'
        return fallback[['content_id', 'title', 'category', 'format', 'popularity_rank_score', 'recommendation_reason']]

    candidates = pd.concat(candidate_frames, ignore_index=True)
    candidates = candidates[~candidates['content_id'].isin(seen_items)].copy()

    if candidates.empty:
        fallback = top_popular.loc[~top_popular['content_id'].isin(seen_items)].head(top_n).copy()
        fallback['recommendation_reason'] = 'Popularity fallback because all similar items were already seen'
        return fallback[['content_id', 'title', 'category', 'format', 'popularity_rank_score', 'recommendation_reason']]

    candidates = (
        candidates.groupby(['content_id', 'title', 'category', 'format', 'mood'], as_index=False)
        .agg(content_similarity_score=('similarity_score', 'mean'))
    )

    candidates = candidates.merge(content_popularity_lookup, on='content_id', how='left')
    candidates['normalized_popularity_score'] = candidates['normalized_popularity_score'].fillna(0)

    user_row = users.loc[users['user_id'] == user_id].iloc[0]
    fav_categories = set(str(user_row['favorite_categories']).split('|'))
    pref_formats = set(str(user_row['preferred_formats']).split('|'))
    pref_moods = set(str(user_row['preferred_moods']).split('|'))

    candidates['profile_match_score'] = (
        candidates['category'].isin(fav_categories).astype(int) * 0.5 +
        candidates['format'].isin(pref_formats).astype(int) * 0.3 +
        candidates['mood'].isin(pref_moods).astype(int) * 0.2
    )

    candidates['final_score'] = (
        0.55 * candidates['content_similarity_score'] +
        0.25 * candidates['normalized_popularity_score'] +
        0.20 * candidates['profile_match_score']
    )

    def reason_builder(row):
        reasons = []
        if row['category'] in fav_categories:
            reasons.append('matches favorite category')
        if row['format'] in pref_formats:
            reasons.append('matches preferred format')
        if row['mood'] in pref_moods:
            reasons.append('fits preferred mood')
        if not reasons:
            reasons.append('similar to previously engaged content')
        return '; '.join(reasons)

    candidates['recommendation_reason'] = candidates.apply(reason_builder, axis=1)

    return candidates.sort_values('final_score', ascending=False).head(top_n).reset_index(drop=True)

sample_user_id = users['user_id'].iloc[0]
recommend_for_user(sample_user_id, top_n=10)


,content_id,title,category,format,mood,content_similarity_score,normalized_popularity_score,profile_match_score,final_score,recommendation_reason
0,C0709,Interesting Workshops and Meetups to Check Out...,Events,Short Video,Social,0.622895,0.785836,0.5,0.639051,matches favorite category
1,C0382,Interesting Workshops and Meetups to Check Out...,Events,Carousel,Curious,0.649362,0.472473,0.7,0.615267,matches favorite category; fits preferred mood
2,C0865,Interesting Workshops and Meetups to Check Out...,Events,Guide,Curious,0.550387,0.666810,0.7,0.609415,matches favorite category; fits preferred mood
3,C0400,Interesting Workshops and Meetups to Check Out...,Events,Article,Social,0.612336,0.684110,0.5,0.607812,matches favorite category
4,C0389,Energetic Tracks for a Better Start to the Day...,Music,Playlist,Energetic,0.530813,0.589334,0.8,0.599281,matches favorite category; matches preferred f...
5,C0307,Interesting Workshops and Meetups to Check Out...,Events,Article,Curious,0.617656,0.469640,0.7,0.597121,matches favorite category; fits preferred mood
6,C0223,Interesting Workshops and Meetups to Check Out...,Events,Carousel,Curious,0.428598,0.819152,0.7,0.580517,matches favorite category; fits preferred mood
7,C0521,Late-Night Tracks for Quiet Study Sessions for...,Music,Playlist,Relaxed,0.409745,0.598891,1.0,0.575083,matches favorite category; matches preferred f...
8,C0479,Energetic Tracks for a Better Start to the Day...,Music,Carousel,Relaxed,0.533757,0.532703,0.7,0.566742,matches favorite category; fits preferred mood
9,C0663,Energetic Tracks for a Better Start to the Day...,Music,Carousel,Relaxed,0.488953,0.624903,0.7,0.565150,matches favorite category; fits preferred mood


## 8. Build sample recommendation outputs

In [11]:
sample_users = users['user_id'].sample(10, random_state=42).tolist()
recommendation_examples = []

for uid in sample_users:
    recs = recommend_for_user(uid, top_n=5).copy()
    recs['user_id'] = uid
    recommendation_examples.append(recs)

recommendation_examples_df = pd.concat(recommendation_examples, ignore_index=True)
recommendation_examples_df.head(15)


,content_id,title,category,format,mood,content_similarity_score,normalized_popularity_score,profile_match_score,final_score,recommendation_reason,user_id
0,C0611,How to Build Better Lecture Notes Without Over...,Study,Guide,Focused,0.498112,0.891608,1.0,0.696864,matches favorite category; matches preferred f...,U0408
1,C0746,How to Build Better Lecture Notes Without Over...,Study,Guide,Focused,0.588520,0.553483,1.0,0.662057,matches favorite category; matches preferred f...,U0408
2,C0457,How to Build Better Lecture Notes Without Over...,Study,Short Video,Calm,0.536601,0.893429,0.7,0.658488,matches favorite category; fits preferred mood,U0408
3,C0110,How to Build Better Lecture Notes Without Over...,Study,Guide,Calm,0.428213,0.819646,1.0,0.640429,matches favorite category; matches preferred f...,U0408
4,C0566,A Practical Guide to Last-Minute Exam Preparat...,Study,Guide,Focused,0.483338,0.604737,1.0,0.617020,matches favorite category; matches preferred f...,U0408
5,C0683,Flashcard Systems Students Actually Keep Using...,Study,Checklist,Focused,0.539492,0.692614,1.0,0.669874,matches favorite category; matches preferred f...,U0445
6,C0037,Flashcard Systems Students Actually Keep Using...,Study,Podcast,Calm,0.515512,0.741385,1.0,0.668878,matches favorite category; matches preferred f...,U0445
7,C0182,Mindfulness Practices That Don’t Feel Performa...,Wellness,Guide,Relaxed,0.605262,0.677374,0.8,0.662237,matches favorite category; matches preferred f...,U0445
8,C0349,Time-Blocking Without Burning Out When Motivat...,Productivity,Podcast,Calm,0.677227,0.661734,0.5,0.637908,matches preferred format; fits preferred mood,U0445
9,C0167,Flashcard Systems Students Actually Keep Using...,Study,Guide,Focused,0.410992,0.782682,1.0,0.621716,matches favorite category; matches preferred f...,U0445


## 9. Save model artifacts

In [12]:
with open(MODELS_DIR / 'tfidf_matrix.pkl', 'wb') as f:
    pickle.dump(tfidf_matrix, f)

with open(MODELS_DIR / 'similarity_matrix.pkl', 'wb') as f:
    pickle.dump(similarity_matrix, f)

recommender_assets = {
    'content_features': content_features,
    'top_popular': top_popular,
    'content_index': content_index.to_dict(),
    'seen_lookup': seen_lookup,
    'users': users,
    'interactions': interaction_scores,
    'user_positive_history': user_positive_history,
    'content_popularity_lookup': content_popularity_lookup,
    'vectorizer': tfidf,
}

with open(MODELS_DIR / 'recommender_assets.pkl', 'wb') as f:
    pickle.dump(recommender_assets, f)

recommendation_examples_df.to_csv(METRICS_DIR / 'sample_recommendations.csv', index=False)
top_popular.head(50).to_csv(METRICS_DIR / 'top_popular_content.csv', index=False)

print('Saved model artifacts and sample recommendation outputs.')


Saved model artifacts and sample recommendation outputs.


## 10. Quick validation checks

In [13]:
print('Processed files created:')
for p in sorted(PROCESSED_DIR.glob('*')):
    print('-', p.name)

print('\nModel files created:')
for p in sorted(MODELS_DIR.glob('*')):
    print('-', p.name)

print('\nMetrics files created:')
for p in sorted(METRICS_DIR.glob('*')):
    print('-', p.name)


Processed files created:
- content_features.csv
- interaction_scores.csv
- user_profiles.csv

Model files created:
- recommender_assets.pkl
- similarity_matrix.pkl
- tfidf_matrix.pkl

Metrics files created:
- category_summary.csv
- persona_summary.csv
- sample_recommendations.csv
- top_popular_content.csv


## 11. Notes for the next notebook

The next notebook should focus on evaluation only.

It can assess:
- recommendation coverage
- category diversity
- average popularity bias
- simple hit-rate style offline evaluation
- examples of recommendations by user persona
